In [ ]:
#Cell 1 — imports, Drive mount, paths (MSVD only, new checkpoint location):
import os
import json
import math
import random
import time
from typing import List, Dict, Any, Tuple
from types import SimpleNamespace

import cv2
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import CLIPModel, CLIPTokenizerFast, get_cosine_schedule_with_warmup

from google.colab import drive
drive.mount('/content/drive')

MSVD_ROOT = "/content/drive/Shareddrives/DATA 298A/DATA/MSVD"

# NEW output dir — deliberately different from the existing output_msvd_clip/
# so this run never overwrites the original best.pt used by 298-32's evaluation.
OUTPUT_DIR = "/content/drive/Shareddrives/DATA 298A/Models/CLIP/output_msvd_clip_v2"

args = SimpleNamespace(
    train_json=os.path.join(MSVD_ROOT, "msvd_train.json"),
    val_json=os.path.join(MSVD_ROOT, "msvd_val.json"),
    video_root=os.path.join(MSVD_ROOT, "raw_videos"),
    output_dir=OUTPUT_DIR,

    model_name="openai/clip-vit-base-patch32",
    image_size=224,
    max_text_len=32,
    num_frames=8,

    batch_size=8,
    num_workers=4,
    epochs=10,
    lr=1e-5,
    weight_decay=1e-4,
    warmup_ratio=0.1,
    grad_accum_steps=2,
    max_grad_norm=1.0,

    freeze_vision=False,
    freeze_text=False,
    train_text_projection_only=False,
    train_vision_projection_only=False,
    unfreeze_last_n_vision=4,
    unfreeze_last_n_text=4,

    save_every=1,
    resume="",
    seed=42,
    amp=True,
    log_every=50,
)

print("train_json exists:", os.path.exists(args.train_json))
print("val_json exists:", os.path.exists(args.val_json))
print("video_root exists:", os.path.exists(args.video_root))
print("output_dir (new):", args.output_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train_json exists: True
val_json exists: True
video_root exists: True
output_dir (new): /content/drive/Shareddrives/DATA 298A/Models/CLIP/output_msvd_clip_v2


In [ ]:
#Cell 2 — all function/class definitions:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def save_json(obj: Dict[str, Any], path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def format_seconds(seconds: float) -> str:
    seconds = int(seconds)
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def build_pairs_from_msvd_json(json_path: str, video_root: str) -> List[Dict[str, str]]:
    with open(json_path, "r", encoding="utf-8") as f:
        records = json.load(f)

    pairs: List[Dict[str, str]] = []
    missing_videos = 0

    for item in records:
        video_name = item["video"]
        captions = item["caption"]
        video_path = os.path.join(video_root, video_name)

        if not os.path.exists(video_path):
            missing_videos += 1
            continue

        if isinstance(captions, str):
            captions = [captions]

        for caption in captions:
            caption = str(caption).strip()
            if caption:
                pairs.append(
                    {
                        "video_id": item["video_id"],
                        "video_path": video_path,
                        "caption": caption,
                    }
                )

    if len(pairs) == 0:
        raise RuntimeError(f"No training pairs found in {json_path}. Check video_root and file names.")

    print(f"Loaded {len(records)} video records from {json_path}")
    print(f"Created {len(pairs)} text-video pairs")
    print(f"Skipped {missing_videos} records because the video file was not found under {video_root}")
    return pairs


class VideoFrameReader:
    def __init__(self, image_size: int, num_frames: int):
        self.image_size = image_size
        self.num_frames = num_frames
        self.transform = transforms.Compose(
            [
                transforms.Resize((image_size, image_size), interpolation=transforms.InterpolationMode.BICUBIC),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=(0.48145466, 0.4578275, 0.40821073),
                    std=(0.26862954, 0.26130258, 0.27577711),
                ),
            ]
        )

    def _sample_indices(self, total_frames: int) -> List[int]:
        if total_frames <= 0:
            return [0] * self.num_frames
        if total_frames < self.num_frames:
            idx = np.linspace(0, total_frames - 1, self.num_frames)
            return [int(round(x)) for x in idx]
        idx = np.linspace(0, total_frames - 1, self.num_frames)
        return [int(x) for x in idx]

    def read_video(self, video_path: str) -> torch.Tensor:
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise RuntimeError(f"Could not open video: {video_path}")

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_indices = self._sample_indices(total_frames)

        frames = []
        for idx in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ok, frame = cap.read()
            if not ok or frame is None:
                if frames:
                    frame_tensor = frames[-1].clone()
                    frames.append(frame_tensor)
                    continue
                else:
                    cap.release()
                    raise RuntimeError(f"Failed to read frame {idx} from {video_path}")
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = Image.fromarray(frame)
            frame_tensor = self.transform(image)
            frames.append(frame_tensor)

        cap.release()
        return torch.stack(frames, dim=0)


class MSVDPairDataset(Dataset):
    def __init__(self, pairs: List[Dict[str, str]], frame_reader: VideoFrameReader):
        self.pairs = pairs
        self.frame_reader = frame_reader

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, index: int) -> Dict[str, Any]:
        item = self.pairs[index]
        frames = self.frame_reader.read_video(item["video_path"])
        return {
            "video_id": item["video_id"],
            "video_path": item["video_path"],
            "caption": item["caption"],
            "frames": frames,
        }


class Batch:
    def __init__(self, pixel_values, captions, video_ids, video_paths):
        self.pixel_values = pixel_values
        self.captions = captions
        self.video_ids = video_ids
        self.video_paths = video_paths


class Collator:
    def __call__(self, batch: List[Dict[str, Any]]) -> Batch:
        pixel_values = torch.stack([x["frames"] for x in batch], dim=0)
        captions = [x["caption"] for x in batch]
        video_ids = [x["video_id"] for x in batch]
        video_paths = [x["video_path"] for x in batch]
        return Batch(pixel_values, captions, video_ids, video_paths)


def extract_tensor_features(output, kind="image"):
    if torch.is_tensor(output):
        return output
    if hasattr(output, "image_embeds") and output.image_embeds is not None:
        return output.image_embeds
    if hasattr(output, "text_embeds") and output.text_embeds is not None:
        return output.text_embeds
    if hasattr(output, "pooler_output") and output.pooler_output is not None:
        return output.pooler_output
    if hasattr(output, "last_hidden_state") and output.last_hidden_state is not None:
        return output.last_hidden_state[:, 0, :]
    if isinstance(output, (tuple, list)) and len(output) > 0:
        if torch.is_tensor(output[0]):
            return output[0]
    raise TypeError(f"Could not extract tensor features from {type(output)}")


class VideoTextCLIP(nn.Module):
    def __init__(self, model_name: str):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(model_name)

    def encode_video(self, pixel_values: torch.Tensor) -> torch.Tensor:
        bsz, num_frames, c, h, w = pixel_values.shape
        flat = pixel_values.view(bsz * num_frames, c, h, w)
        image_features = self.clip.get_image_features(pixel_values=flat)
        image_features = extract_tensor_features(image_features, kind="image")
        image_features = image_features.view(bsz, num_frames, -1)
        video_features = image_features.mean(dim=1)
        video_features = F.normalize(video_features, dim=-1)
        return video_features

    def encode_text(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        text_features = self.clip.get_text_features(input_ids=input_ids, attention_mask=attention_mask)
        text_features = extract_tensor_features(text_features, kind="text")
        text_features = F.normalize(text_features, dim=-1)
        return text_features

    def forward(self, pixel_values, input_ids, attention_mask):
        video_features = self.encode_video(pixel_values)
        text_features = self.encode_text(input_ids, attention_mask)
        logit_scale = self.clip.logit_scale.exp().clamp(max=100.0)
        return video_features, text_features, logit_scale


def freeze_module(module: nn.Module) -> None:
    for p in module.parameters():
        p.requires_grad = False


def unfreeze_module(module: nn.Module) -> None:
    for p in module.parameters():
        p.requires_grad = True


def apply_finetune_policy(model, args) -> None:
    freeze_module(model)
    model.clip.logit_scale.requires_grad = True

    if args.train_text_projection_only:
        unfreeze_module(model.clip.text_projection)
        return
    if args.train_vision_projection_only:
        unfreeze_module(model.clip.visual_projection)
        return

    if not args.freeze_vision:
        unfreeze_module(model.clip.vision_model.post_layernorm)
        unfreeze_module(model.clip.visual_projection)
        vision_layers = model.clip.vision_model.encoder.layers
        n = min(args.unfreeze_last_n_vision, len(vision_layers))
        for layer in vision_layers[-n:]:
            unfreeze_module(layer)

    if not args.freeze_text:
        unfreeze_module(model.clip.text_model.final_layer_norm)
        unfreeze_module(model.clip.text_projection)
        text_layers = model.clip.text_model.encoder.layers
        n = min(args.unfreeze_last_n_text, len(text_layers))
        for layer in text_layers[-n:]:
            unfreeze_module(layer)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {trainable:,} / {total:,}")


def contrastive_loss(video_features, text_features, logit_scale) -> torch.Tensor:
    logits_per_video = logit_scale * video_features @ text_features.t()
    logits_per_text = logits_per_video.t()
    targets = torch.arange(video_features.size(0), device=video_features.device)
    loss_v2t = F.cross_entropy(logits_per_video, targets)
    loss_t2v = F.cross_entropy(logits_per_text, targets)
    return 0.5 * (loss_v2t + loss_t2v)


def build_optimizer(model, lr, weight_decay):
    decay_params, no_decay_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim < 2 or name.endswith("bias") or "norm" in name.lower():
            no_decay_params.append(p)
        else:
            decay_params.append(p)
    return torch.optim.AdamW(
        [
            {"params": decay_params, "weight_decay": weight_decay},
            {"params": no_decay_params, "weight_decay": 0.0},
        ],
        lr=lr, betas=(0.9, 0.98), eps=1e-6,
    )


def tokenize_text(tokenizer, captions, max_text_len, device):
    encoded = tokenizer(captions, padding=True, truncation=True, max_length=max_text_len, return_tensors="pt")
    return {k: v.to(device) for k, v in encoded.items()}


def run_train_epoch(model, tokenizer, loader, optimizer, scheduler, scaler, device, epoch, args) -> float:
    model.train()
    running_loss = 0.0
    step_count = 0
    optimizer.zero_grad(set_to_none=True)
    start_time = time.time()

    for step, batch in enumerate(loader):
        pixel_values = batch.pixel_values.to(device, non_blocking=True)
        text_inputs = tokenize_text(tokenizer, batch.captions, args.max_text_len, device)

        use_amp = args.amp and device.type == "cuda"
        with torch.cuda.amp.autocast(enabled=use_amp):
            video_features, text_features, logit_scale = model(
                pixel_values=pixel_values,
                input_ids=text_inputs["input_ids"],
                attention_mask=text_inputs["attention_mask"],
            )
            loss = contrastive_loss(video_features, text_features, logit_scale)
            loss = loss / args.grad_accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % args.grad_accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        running_loss += loss.item() * args.grad_accum_steps
        step_count += 1

        if (step + 1) % args.log_every == 0:
            elapsed = time.time() - start_time
            avg_loss = running_loss / step_count
            print(f"[Train] Epoch {epoch} Step {step + 1}/{len(loader)} Loss {avg_loss:.4f} LR {scheduler.get_last_lr()[0]:.7f} Elapsed {format_seconds(elapsed)}")

    return running_loss / max(step_count, 1)


@torch.no_grad()
def run_val_epoch(model, tokenizer, loader, device, args, epoch) -> float:
    model.eval()
    running_loss = 0.0
    step_count = 0

    for step, batch in enumerate(loader):
        pixel_values = batch.pixel_values.to(device, non_blocking=True)
        text_inputs = tokenize_text(tokenizer, batch.captions, args.max_text_len, device)

        use_amp = args.amp and device.type == "cuda"
        with torch.cuda.amp.autocast(enabled=use_amp):
            video_features, text_features, logit_scale = model(
                pixel_values=pixel_values,
                input_ids=text_inputs["input_ids"],
                attention_mask=text_inputs["attention_mask"],
            )
            loss = contrastive_loss(video_features, text_features, logit_scale)

        running_loss += loss.item()
        step_count += 1

    avg_loss = running_loss / max(step_count, 1)
    print(f"[Val] Epoch {epoch} Loss {avg_loss:.4f}")
    return avg_loss


def save_checkpoint(path, model, optimizer, scheduler, scaler, epoch, best_val_loss, args) -> None:
    state = {
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "best_val_loss": best_val_loss,
        "args": vars(args),
    }
    torch.save(state, path)
    print(f"Saved checkpoint to {path}")


print("All functions and classes defined")

All functions and classes defined


In [ ]:
#Cell 3 — build train/val pairs, confirm counts before starting:
ensure_dir(args.output_dir)
set_seed(args.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

train_pairs = build_pairs_from_msvd_json(args.train_json, args.video_root)
val_pairs = build_pairs_from_msvd_json(args.val_json, args.video_root)

print(f"\nTrain pairs: {len(train_pairs)}")
print(f"Val pairs: {len(val_pairs)}")
print("Sample train pair:", train_pairs[0])

Using device: cuda
Loaded 1200 video records from /content/drive/Shareddrives/DATA 298A/DATA/MSVD/msvd_train.json
Created 48774 text-video pairs
Skipped 0 records because the video file was not found under /content/drive/Shareddrives/DATA 298A/DATA/MSVD/raw_videos
Loaded 100 video records from /content/drive/Shareddrives/DATA 298A/DATA/MSVD/msvd_val.json
Created 4290 text-video pairs
Skipped 0 records because the video file was not found under /content/drive/Shareddrives/DATA 298A/DATA/MSVD/raw_videos

Train pairs: 48774
Val pairs: 4290
Sample train pair: {'video_id': 'WTf5EgVY5uU_98_104', 'video_path': '/content/drive/Shareddrives/DATA 298A/DATA/MSVD/raw_videos/WTf5EgVY5uU_98_104.avi', 'caption': 'a woman breaks an egg'}


In [ ]:
#Cell 4 — build model, dataloaders, optimizer, scheduler (quick, no long wait):
train_dataset = MSVDPairDataset(train_pairs, VideoFrameReader(image_size=args.image_size, num_frames=args.num_frames))
val_dataset = MSVDPairDataset(val_pairs, VideoFrameReader(image_size=args.image_size, num_frames=args.num_frames))

collator = Collator()
train_loader = DataLoader(
    train_dataset, batch_size=args.batch_size, shuffle=True,
    num_workers=args.num_workers, pin_memory=True, drop_last=True,
    collate_fn=collator, persistent_workers=args.num_workers > 0,
)
val_loader = DataLoader(
    val_dataset, batch_size=args.batch_size, shuffle=False,
    num_workers=args.num_workers, pin_memory=True, drop_last=False,
    collate_fn=collator, persistent_workers=args.num_workers > 0,
)

tokenizer = CLIPTokenizerFast.from_pretrained(args.model_name)
model = VideoTextCLIP(args.model_name)
apply_finetune_policy(model, args)
model.to(device)

optimizer = build_optimizer(model, lr=args.lr, weight_decay=args.weight_decay)
total_update_steps = math.ceil(len(train_loader) / args.grad_accum_steps) * args.epochs
warmup_steps = int(total_update_steps * args.warmup_ratio)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_update_steps)
scaler = torch.cuda.amp.GradScaler(enabled=args.amp and device.type == "cuda")

print(f"Steps per epoch: {len(train_loader)}")
print(f"Total update steps: {total_update_steps}")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Trainable parameters: 41,618,945 / 151,277,313
Steps per epoch: 6096
Total update steps: 30480


/tmp/ipykernel_7875/1279615865.py:26: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=args.amp and device.type == "cuda")


In [5]:
#Cell 5 — the actual training loop (this is the long-running one):
start_epoch, best_val_loss = 1, float("inf")

save_json(vars(args), os.path.join(args.output_dir, "train_args.json"))

history = []
train_start = time.time()

for epoch in range(start_epoch, args.epochs + 1):
    epoch_start = time.time()

    train_loss = run_train_epoch(
        model=model, tokenizer=tokenizer, loader=train_loader,
        optimizer=optimizer, scheduler=scheduler, scaler=scaler,
        device=device, epoch=epoch, args=args,
    )

    val_loss = run_val_epoch(
        model=model, tokenizer=tokenizer, loader=val_loader,
        device=device, args=args, epoch=epoch,
    )

    epoch_time = time.time() - epoch_start
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "epoch_time_sec": epoch_time,
    })
    save_json(history, os.path.join(args.output_dir, "history.json"))

    print(f"Epoch {epoch} done | Train Loss {train_loss:.4f} | Val Loss {val_loss:.4f} | Time {format_seconds(epoch_time)}")

    if epoch % args.save_every == 0:
        ckpt_path = os.path.join(args.output_dir, f"checkpoint_epoch_{epoch}.pt")
        save_checkpoint(ckpt_path, model, optimizer, scheduler, scaler, epoch, best_val_loss, args)

    last_path = os.path.join(args.output_dir, "last.pt")
    save_checkpoint(last_path, model, optimizer, scheduler, scaler, epoch, best_val_loss, args)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_path = os.path.join(args.output_dir, "best.pt")
        save_checkpoint(best_path, model, optimizer, scheduler, scaler, epoch, best_val_loss, args)
        print(f"New best checkpoint at epoch {epoch} with val loss {val_loss:.4f}")

total_time = time.time() - train_start
print(f"\nTraining finished in {format_seconds(total_time)}")
print(f"Best val loss: {best_val_loss:.4f}")

/tmp/ipykernel_7875/2766318587.py:285: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):
/tmp/ipykernel_7875/2766318587.py:302: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


[Train] Epoch 1 Step 50/6096 Loss 0.3505 LR 0.0000001 Elapsed 00:00:59
[Train] Epoch 1 Step 100/6096 Loss 0.3379 LR 0.0000002 Elapsed 00:01:54
[Train] Epoch 1 Step 150/6096 Loss 0.3448 LR 0.0000002 Elapsed 00:02:36
[Train] Epoch 1 Step 200/6096 Loss 0.3621 LR 0.0000003 Elapsed 00:03:16
[Train] Epoch 1 Step 250/6096 Loss 0.3596 LR 0.0000004 Elapsed 00:03:55
[Train] Epoch 1 Step 300/6096 Loss 0.3542 LR 0.0000005 Elapsed 00:04:32
[Train] Epoch 1 Step 350/6096 Loss 0.3393 LR 0.0000006 Elapsed 00:05:08
[Train] Epoch 1 Step 400/6096 Loss 0.3313 LR 0.0000007 Elapsed 00:05:44
[Train] Epoch 1 Step 450/6096 Loss 0.3249 LR 0.0000007 Elapsed 00:06:18
[Train] Epoch 1 Step 500/6096 Loss 0.3213 LR 0.0000008 Elapsed 00:06:54
[Train] Epoch 1 Step 550/6096 Loss 0.3165 LR 0.0000009 Elapsed 00:07:30
[Train] Epoch 1 Step 600/6096 Loss 0.3107 LR 0.0000010 Elapsed 00:08:04
[Train] Epoch 1 Step 650/6096 Loss 0.3071 LR 0.0000011 Elapsed 00:08:40
[Train] Epoch 1 Step 700/6096 Loss 0.2974 LR 0.0000011 Elapsed 00

/tmp/ipykernel_7875/2766318587.py:326: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[Val] Epoch 1 Loss 2.7895
Epoch 1 done | Train Loss 0.1719 | Val Loss 2.7895 | Time 01:15:22
Saved checkpoint to /content/drive/Shareddrives/DATA 298A/Models/CLIP/output_msvd_clip_v2/checkpoint_epoch_1.pt
Saved checkpoint to /content/drive/Shareddrives/DATA 298A/Models/CLIP/output_msvd_clip_v2/last.pt
Saved checkpoint to /content/drive/Shareddrives/DATA 298A/Models/CLIP/output_msvd_clip_v2/best.pt
New best checkpoint at epoch 1 with val loss 2.7895
[Train] Epoch 2 Step 50/6096 Loss 0.0830 LR 0.0000100 Elapsed 00:00:37
[Train] Epoch 2 Step 100/6096 Loss 0.0816 LR 0.0000100 Elapsed 00:01:10
[Train] Epoch 2 Step 150/6096 Loss 0.0939 LR 0.0000100 Elapsed 00:01:45
[Train] Epoch 2 Step 200/6096 Loss 0.0892 LR 0.0000100 Elapsed 00:02:20
[Train] Epoch 2 Step 250/6096 Loss 0.0961 LR 0.0000100 Elapsed 00:02:53
[Train] Epoch 2 Step 300/6096 Loss 0.0899 LR 0.0000100 Elapsed 00:03:28
[Train] Epoch 2 Step 350/6096 Loss 0.0986 LR 0.0000100 Elapsed 00:03:57
[Train] Epoch 2 Step 400/6096 Loss 0.1010 LR

KeyboardInterrupt: 

In [6]:
#Cell 6 — evaluate the new epoch-1 checkpoint (reusing the same eval pattern from 298-32):

!pip install decord
from transformers import CLIPProcessor
from decord import VideoReader, cpu

NEW_CHECKPOINT = os.path.join(OUTPUT_DIR, "best.pt")
print("New checkpoint exists:", os.path.exists(NEW_CHECKPOINT))

class CLIPVideoTextModel(nn.Module):
    def __init__(self, model_name: str):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(model_name)

    def encode_text(self, input_ids, attention_mask):
        text_features = self.clip.get_text_features(input_ids=input_ids, attention_mask=attention_mask)
        text_features = extract_tensor_features(text_features, kind="text")
        return F.normalize(text_features, dim=-1)

    def encode_video(self, pixel_values):
        bsz, t, c, h, w = pixel_values.shape
        flat_pixels = pixel_values.reshape(bsz * t, c, h, w)
        frame_features = self.clip.get_image_features(pixel_values=flat_pixels)
        frame_features = extract_tensor_features(frame_features, kind="image")
        frame_features = frame_features.reshape(bsz, t, -1)
        video_features = frame_features.mean(dim=1)
        return F.normalize(video_features, dim=-1)


def load_video_frames_eval(video_path, num_frames):
    vr = VideoReader(video_path, ctx=cpu(0))
    idxs = np.linspace(0, len(vr) - 1, num_frames, dtype=int).tolist()
    frames = vr.get_batch(idxs).asnumpy()
    return [Image.fromarray(f).convert("RGB") for f in frames]


def build_test_set_msvd_eval(data_root):
    test_file = os.path.join(data_root, "msvd_test.json")
    video_root = os.path.join(data_root, "raw_videos")
    with open(test_file, "r", encoding="utf-8") as f:
        records = json.load(f)
    videos, queries = {}, []
    for item in records:
        video_name = item["video"]
        video_id = item.get("video_id", video_name)
        video_path = os.path.join(video_root, video_name)
        if not os.path.exists(video_path):
            continue
        if video_id not in videos:
            videos[video_id] = {"video_id": video_id, "video_path": video_path}
        captions = item["caption"]
        if isinstance(captions, str):
            captions = [captions]
        for cap in captions:
            cap = " ".join(str(cap).strip().split())
            if cap:
                queries.append({"video_id": video_id, "caption": cap})
    return queries, list(videos.values())


eval_processor = CLIPProcessor.from_pretrained(args.model_name)
eval_model = CLIPVideoTextModel(args.model_name).to(device)

ckpt = torch.load(NEW_CHECKPOINT, map_location="cpu")
eval_model.load_state_dict(ckpt["model"], strict=False)
eval_model.eval()
print(f"Loaded new checkpoint from epoch {ckpt['epoch']}, best_val_loss={ckpt['best_val_loss']:.4f}")

eval_queries, eval_videos = build_test_set_msvd_eval(MSVD_ROOT)
print(f"Eval — Queries: {len(eval_queries)} | Videos: {len(eval_videos)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 139.5 MB/s eta 0:00:00
New checkpoint exists: True


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loaded new checkpoint from epoch 1, best_val_loss=2.7895
Eval — Queries: 27763 | Videos: 670


In [7]:
#Cell 7 — encode + compute metrics:
@torch.no_grad()
def encode_videos_eval(model, processor, videos, device, num_frames, batch_size):
    all_embeds, video_ids = [], []
    for start in range(0, len(videos), batch_size):
        batch = videos[start:start + batch_size]
        frames_nested = [load_video_frames_eval(v["video_path"], num_frames) for v in batch]
        flat_images = [img for frames in frames_nested for img in frames]
        image_inputs = processor(images=flat_images, return_tensors="pt")
        pixel_values = image_inputs["pixel_values"].reshape(len(batch), num_frames, *image_inputs["pixel_values"].shape[1:]).to(device)
        video_embeds = model.encode_video(pixel_values)
        all_embeds.append(video_embeds.cpu())
        video_ids.extend([v["video_id"] for v in batch])
        if (start // batch_size) % 10 == 0:
            print(f"Encoded {start + len(batch)}/{len(videos)} videos")
    return torch.cat(all_embeds, dim=0), video_ids


@torch.no_grad()
def encode_queries_eval(model, processor, queries, device, max_text_len, batch_size):
    all_embeds, gt_video_ids = [], []
    for start in range(0, len(queries), batch_size):
        batch = queries[start:start + batch_size]
        texts = [q["caption"] for q in batch]
        gt_video_ids.extend([q["video_id"] for q in batch])
        text_inputs = processor(text=texts, padding=True, truncation=True, max_length=max_text_len, return_tensors="pt")
        text_embeds = model.encode_text(text_inputs["input_ids"].to(device), text_inputs["attention_mask"].to(device))
        all_embeds.append(text_embeds.cpu())
    return torch.cat(all_embeds, dim=0), gt_video_ids


def compute_metrics_eval(similarity, gt_video_ids, video_ids):
    video_id_to_idx = {vid: i for i, vid in enumerate(video_ids)}
    gt_indices = np.array([video_id_to_idx[v] for v in gt_video_ids])
    sorted_idx = np.argsort(-similarity, axis=1)
    ranks = np.array([int(np.where(sorted_idx[i] == gt_indices[i])[0][0]) + 1 for i in range(len(gt_indices))])
    return {
        "R@1": float(np.mean(ranks <= 1)), "R@5": float(np.mean(ranks <= 5)), "R@10": float(np.mean(ranks <= 10)),
        "MRR": float(np.mean(1.0 / ranks)), "MeanRank": float(np.mean(ranks)), "MedianRank": float(np.median(ranks)),
    }


import time
start_time = time.time()
video_embeds_new, video_ids_new = encode_videos_eval(eval_model, eval_processor, eval_videos, device, 8, 16)
text_embeds_new, gt_ids_new = encode_queries_eval(eval_model, eval_processor, eval_queries, device, 32, 64)
encode_time_new = time.time() - start_time

similarity_new = (text_embeds_new @ video_embeds_new.T).numpy()
metrics_new = compute_metrics_eval(similarity_new, gt_ids_new, video_ids_new)
metrics_new["avg_query_latency_sec"] = float(encode_time_new / max(len(eval_queries), 1))

print("\nNew fine-tuned CLIP (epoch 1) — MSVD Retrieval Metrics")
for k, v in metrics_new.items():
    print(f"{k}: {v}")

Encoded 16/670 videos
Encoded 176/670 videos
Encoded 336/670 videos
Encoded 496/670 videos
Encoded 656/670 videos

New fine-tuned CLIP (epoch 1) — MSVD Retrieval Metrics
R@1: 0.3648380938659367
R@5: 0.6646255808089904
R@10: 0.7740157763930411
MRR: 0.5018587867848034
MeanRank: 15.249504736519828
MedianRank: 2.0
avg_query_latency_sec: 0.01614055593011273


In [8]:
#Cell 8 — save results to download:
final_results = {
    "model": "CLIP",
    "variant": "fine_tuned_v2",
    "dataset": "MSVD",
    "checkpoint": NEW_CHECKPOINT,
    "training_epoch": ckpt["epoch"],
    "best_val_loss": ckpt["best_val_loss"],
    "metrics": metrics_new,
}

with open("/content/clip_finetuned_v2_results.json", "w") as f:
    json.dump(final_results, f, indent=2)

print("Saved.")

Saved.
